In [1]:
import os
import torch

# Disable compile path that can fail on Phi longrope dynamic branching.
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # auto-detect (float16 on most GPUs)
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0305 01:12:44.668000 33668 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Phi3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5060 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [3]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files="finetune_messages_train.jsonl", split="train")
val_dataset = load_dataset("json", data_files="finetune_messages_val.jsonl", split="train")

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(train_dataset[0])

Train samples: 2400
Val samples:   600
{'messages': [{'content': '\n\nAnswer in hinglish only', 'role': 'system'}, {'content': 'Bharat ke 5 famous linguists aur unke latest research projects ko samjha kar share kijiye.', 'role': 'user'}, {'content': '1. Dr. Ganesh N. Devy: Dr. Devy is a renowned linguist and activist, known for his work on Indian languages and tribal dialects. His latest project is "Bhasha Sangrah," a comprehensive documentation of India\'s linguistic diversity, which includes compiling a list of 780 languages and dialects spoken in the country.\n\n2. Prof. Rajendra Singh: Prof. Singh, also known as "Rajinder Singhji," is a linguist specializing in Sanskrit and comparative linguistics. His recent research focuses on the evolution of Indo-Aryan languages and their connections with other language families. He has also worked on digitizing ancient Indian texts.\n\n3. Dr. Priti Shah: Dr. Shah is a linguist and educationist who works on multilingual education and language p

In [4]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chat)
val_dataset = val_dataset.map(format_chat)

print(train_dataset[0]["text"][:500])

<|system|>

Answer in hinglish only<|end|><|user|>Bharat ke 5 famous linguists aur unke latest research projects ko samjha kar share kijiye.<|end|><|assistant|>1. Dr. Ganesh N. Devy: Dr. Devy is a renowned linguist and activist, known for his work on Indian languages and tribal dialects. His latest project is "Bhasha Sangrah," a comprehensive documentation of India's linguistic diversity, which includes compiling a list of 780 languages and dialects spoken in the country.

2. Prof. Rajendra Sing


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/2400 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/600 [00:00<?, ? examples/s]

In [6]:
gpu_stats = torch.cuda.get_device_properties(0)
reserved_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_memory} GB")
print(f"Reserved VRAM: {reserved_memory} GB")

GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Total VRAM: 7.96 GB
Reserved VRAM: 3.07 GB


In [ ]:
# Train in eager mode to avoid TorchDynamo graph breaks in Phi longrope.
trainer_stats = trainer.train()
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,400 | Num Epochs = 2 | Total steps = 600
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


In [ ]:
peak_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"Peak reserved VRAM: {peak_memory} GB")
print(f"Training runtime: {round(trainer_stats.metrics['train_runtime'], 2)}s")

Peak reserved VRAM: 4.94 GB
Training runtime: 3.84s


In [ ]:
# Save LoRA adapter only
model.save_pretrained("phi4-mini-phinglish-lora")
tokenizer.save_pretrained("phi4-mini-phinglish-lora")

# Save merged 16-bit model for inference
model.save_pretrained_merged("phi4-mini-phinglish-merged", tokenizer, save_method="merged_16bit")

Found HuggingFace hub cache directory: C:\Users\highk\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [09:39<09:39, 579.07s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [21:22<00:00, 641.06s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:08<00:00,  4.23s/it]


Unsloth: Merge process complete. Saved to `d:\Projects\Phinglish\phi4-mini-phinglish-gguf`


In [ ]:
with open("training_log.txt", "w") as f:
    f.write(f"GPU: {gpu_stats.name}\n")
    f.write(f"Total VRAM: {total_memory} GB\n")
    f.write(f"Reserved VRAM: {reserved_memory} GB\n")
    f.write(f"Peak reserved VRAM: {peak_memory} GB\n")
    f.write(f"Training runtime: {round(trainer_stats.metrics['train_runtime'], 2)}s\n")

with open("training_stats.txt", "w") as f:
    for key, value in trainer_stats.metrics.items():
        f.write(f"{key}: {value}\n")

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an AI assistant. Answer in mix of hindi and hinglish"},
    {"role": "user", "content": "India mein best street food kya hai?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


You are an AI assistant. Answer in mix of hindi and hinglishIndia mein best street food kya hai?India mein street food ek zyada popular hai. Ek main street food items jo ekas mein milta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki ekas mein milne lagta hai, jabki
